# 计划 33：C3/C4 外部主验证

在 Kaggle GPU 上运行 `configs/internal/33_external_c3_c4_main_validation.json`。保存版本前，只需挂载以下 Kaggle 数据集：

- BMS 原始 Excel 数据（已有 `dataset-bms`）

MSL 原始 `.npy` 与 NASA Random Discharge 的 `RW*.mat` 已随 Git 代码拉取，下面的预处理单元会将它们转换为训练用 pkl。当前挂载的 `TSINGHUA_EV` 不参与计划 33。

预处理产物和实验输出只会写到 `/kaggle/working/EnhancedMTADGAT`。

In [ ]:
from pathlib import Path
import os
import sys
import torch

# 在启动任何长训练前确认 Kaggle 已启用 GPU。
assert torch.cuda.is_available(), '请在 Settings → Accelerator 中启用 GPU（T4/P100）。'
print('Python 版本:', sys.version.split()[0])
print('PyTorch 版本:', torch.__version__)
print('GPU 名称:', torch.cuda.get_device_name(0))
print('CUDA 版本:', torch.version.cuda)

input_root = Path('/kaggle/input')
for mounted in sorted(path for path in input_root.iterdir() if path.is_dir()):
    files = sum(1 for item in mounted.rglob('*') if item.is_file())
    print(f'输入数据  {mounted}  （{files} 个文件）')

In [ ]:
# 进入 Kaggle 可写工作目录。
%cd /kaggle/working
# 清除同名旧代码副本；只作用于 Kaggle 临时工作目录。
!rm -rf /kaggle/working/EnhancedMTADGAT
# 从 GitHub 拉取刚推送的 main 分支最新代码；depth=1 不下载旧提交历史。
!git clone --depth 1 https://github.com/wonkawonka/EnhancedMTADGAT.git
# 进入项目目录。


%cd /kaggle/working/EnhancedMTADGAT
# 打印当前代码提交号，便于结果追溯。
!git log -1 --oneline
# 安装主线训练所需的 Python 依赖。
!pip install -q -r requirements-kaggle-main.txt

In [ ]:
# 强制将可写的预处理数据和实验输出放在 Kaggle 工作目录，避免写入只读的 /kaggle/input。
os.environ['MTAD_GAT_DATASETS_ROOT'] = '/kaggle/working/EnhancedMTADGAT/datasets'
os.environ['MTAD_GAT_RUNS_ROOT'] = '/kaggle/working/EnhancedMTADGAT/runs'

# BMS 是 Kaggle Input；先从实际挂载目录定位原始 Excel，再显式传给数据路径解析器。
bms_raw_files = list(Path('/kaggle/input').rglob('*_BMS0Data.xls')) + list(Path('/kaggle/input').rglob('*_BMS0Data.xlsx'))
if not bms_raw_files:
    raise FileNotFoundError('未找到 BMS 原始 Excel。请确认 notebook 已挂载 dataset-BMS。')
os.environ['MTAD_GAT_BMS_ROOT'] = str(bms_raw_files[0].parent)
print('BMS 挂载目录:', os.environ['MTAD_GAT_BMS_ROOT'])

from src.project_paths import resolve_dataset_root
roots = {
    'MSL': resolve_dataset_root('DATA', 'data'),
    'NASA_RANDOM_DISCHARGE': resolve_dataset_root('NASA_RANDOM_DISCHARGE', 'NASA_RANDOM_DISCHARGE'),
    'BMS': resolve_dataset_root('BMS', 'BMS'),
}
for name, root in roots.items():
    print(f'{name} 数据根目录: {root}')

missing = []
msl_root = roots['MSL']
if not (msl_root / 'labeled_anomalies.csv').is_file() or not (msl_root / 'train').is_dir() or not (msl_root / 'test').is_dir():
    missing.append('MSL/SMAP：labeled_anomalies.csv + train/ + test/')
nasa_root = roots['NASA_RANDOM_DISCHARGE']
required_rw = ['RW1', 'RW2', 'RW7', 'RW8']
if not all(any(nasa_root.rglob(f'{battery}.mat')) for battery in required_rw):
    missing.append('NASA Random Discharge：RW1.mat、RW2.mat、RW7.mat、RW8.mat')
bms_root = roots['BMS']
if not any(bms_root.glob('*_BMS0Data.xls')) and not any(bms_root.glob('*_BMS0Data.xlsx')):
    missing.append('BMS 原始 Excel：*_BMS0Data.xls/xlsx 及对应 stat/temp/volt 文件')

if missing:
    raise FileNotFoundError('运行前请挂载缺失的 Kaggle 数据集：\n- ' + '\n- '.join(missing))
print('计划 33 所需的全部输入数据已找到。')

In [ ]:
# 分别预处理代码自带的 MSL/NASA 和 Kaggle Input 中的 BMS；生成的 pkl 写入工作目录。
# 将 MSL 的原始序列和标签转换为含边界信息的 pkl。
!python run.py preprocess --dataset MSL
# 将 NASA Random Discharge 的 RW*.mat 转换为训练 pkl。
!python run.py preprocess --dataset NASA_RANDOM_DISCHARGE
# 将 BMS 原始 Excel 按簇连续片段转换为训练 pkl。
!python run.py preprocess --dataset BMS

In [ ]:
# 长时间 GPU 训练前先干跑，确认计划展开后的命令与数据路径。
!python run.py internal --plan configs/internal/33_external_c3_c4_main_validation.json --resume --skip-existing --dry-run

In [ ]:
# 正式运行计划 33：按随机种子展开后共 14 组实验；中断后可用 resume/skip-existing 续跑。
!python run.py internal --plan configs/internal/33_external_c3_c4_main_validation.json --resume --skip-existing

In [ ]:
from pathlib import Path
run_root = Path('runs/internal/33_external_c3_c4_main_validation')
print('实验输出根目录:', run_root.resolve())
for path in sorted(run_root.rglob('*.json'))[:80]:
    print(path)
print('Kaggle 保存版本后会将 /kaggle/working 作为该版本的输出。')